# Modelado Predictivo de la Infracción C32

Se desarrolla un modelo predictivo para la infracción C32 (no respetar el paso de peatones que cruzan una vía en sitio permitido para ellos o no darles la prelación en las franjas para ello establecidas) empleando una metodología iterativa en la que cada nuevo modelo aprende de los resultados obtenidos por los modelos previamente entrenados. El proceso inicia con el modelo Holt-Winters, cuyos parámetros se ajustan con base en los hallazgos del análisis exploratorio y la descomposición STL de la serie temporal. Posteriormente, se incorporan secuencialmente los modelos ARIMA, SARIMA, Dynamic Optimized Theta, Ridge, Lasso, Random Forest, XGBoost, LightGBM y KNN, de modo que cada uno aprovecha el conocimiento acumulado por sus predecesores, con el objetivo de mejorar progresivamente el rendimiento predictivo. Las métricas de evaluación utilizadas son RMSE, MAE, MAPE, SMAPE y MSE. Finalmente, se selecciona el mejor modelo para realizar pronósticos del volumen de infracciones por no respetar el paso de peatones para el año 2026, cuyos datos aún no han sido publicados por la Alcaldía de Barranquilla a través de su portal de Datos Abiertos.

## Carga de librerías

Se importan las librerías necesarias para el desarrollo de modelos predictivos sobre las series temporales de comparendos electrónicos. Se emplean `pandas` y `numpy` para la manipulación y transformación de datos, `plotly.graph_objects` y `plotly.subplots` para la visualización de resultados, y `statsmodels.tsa` para los modelos estadísticos clásicos (Holt-Winters, ARIMA, SARIMA y STL). Para los enfoques de machine learning, se utilizan `scikit-learn` con `RidgeCV`, `MultiTaskLassoCV`, `RandomForestRegressor`, `KNeighborsRegressor` y `GridSearchCV` para optimización de hiperparámetros, junto con `xgboost` (XGBoost) y `lightgbm` (LightGBM). Adicionalmente, se incorporan `sklearn.model_selection.TimeSeriesSplit` para validación con series temporales, `sklearn.preprocessing.StandardScaler` para estandarización de características, y `sklearn.metrics` para las métricas de evaluación (RMSE, MAE, MAPE, SMAPE y MSE). Se suprimen los mensajes de advertencia para mantener una salida limpia y enfocada en los resultados.

In [1]:
import warnings

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import STL
from scipy import stats as sp_stats
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import MultiTaskLassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.neighbors import KNeighborsRegressor

warnings.filterwarnings('ignore')

In [2]:
pio.renderers.default = "notebook_connected"

## Carga del DataFrame

Se carga el archivo CSV que contiene los datos de comparendos electrónicos utilizando la función `read_csv` de la librería pandas, y se almacena en el DataFrame `df_comparendos_electronicos`.

In [3]:
df_comparendos_electronicos = pd.read_csv("C:/Users/david/Documents/seminario_investigativo/comparendos_electronicos.csv")

### Conversión de las Fechas a Formato datetime

Se convierte la columna `fecha_comparendo` al tipo `datetime` utilizando el formato especificado (`'%Y %b %d %I:%M:%S %p'`), normalizando la hora a medianoche con el método `.dt.normalize()` para eliminar la información horaria y trabajar únicamente con fechas, dado que todos los registros contienen la hora de 12:00:00 AM. Posteriormente, se imprime el tipo de dato resultante para verificar la correcta conversión.

In [4]:
df_comparendos_electronicos['fecha_comparendo'] = pd.to_datetime(df_comparendos_electronicos['fecha_comparendo'], format='%Y %b %d %I:%M:%S %p').dt.normalize()

print(df_comparendos_electronicos['fecha_comparendo'].dtype)

datetime64[ns]


### Corrección de Valores Nulos

Se rellenan los valores nulos de la columna `DESC_INFRACCION` con la descripción correspondiente al código C14, obtenida del sitio web oficial del Tránsito del Atlántico. Según esta fuente, la infracción C14 corresponde a **"Transitar por sitios restringidos o en horas prohibidas por la autoridad competente"**. Esta corrección se aplica exclusivamente a los 564 registros que presentaban valores nulos, asociados a las nuevas cámaras tipo Carril Bus implementadas en Barranquilla a partir del 17 de octubre de 2025.

**Fuente:** https://transitodelatlantico.gov.co/valor-de-multas-de-transito/

In [5]:
df_comparendos_electronicos['DESC_INFRACCION'] = df_comparendos_electronicos['DESC_INFRACCION'].fillna("Transitar por sitios restringidos o en horas prohibidas por la autoridad competente")  

print(f"Total de valores nulos en el DataFrame: {df_comparendos_electronicos.isnull().sum().sum()}")

Total de valores nulos en el DataFrame: 0


### Corrección de Cámaras Duplicadas

Se unifican los nombres de las cámaras que presentan dos notaciones diferentes para el mismo punto de control. Esta corrección permite evitar la duplicación de ubicaciones en los análisis y garantizar la consistencia de los datos.

In [6]:
df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CARRERA 53 CON CALLE 104', 'Camara_y_direccion'] = 'CARRERA 53 ENTRE CALLE 104 Y 106'

df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CALLE 84 CON CARRERA 59', 'Camara_y_direccion'] = 'CALLE 84 ENTRE CARRERA 59 Y 59B'

df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CARRERA 45 CON CALLE 53', 'Camara_y_direccion'] = 'CALLE 53 CON CARRERA 45'

df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CALLE 45B CARRERA 14', 'Camara_y_direccion'] = 'CALLE 45B CON CARRERA 14'

### Corrección de Infracciones C02 Detectadas por Cámaras Fijas

Dado que estos registros representan una inconsistencia en la base de datos (las cámaras fijas no están diseñadas para detectar estacionamiento prohibido), se procede a eliminarlos del DataFrame principal para garantizar la consistencia de los análisis posteriores. Posteriormente, se verifica que no queden registros residuales de C02 asociados a cámaras fijas, confirmando la correcta limpieza de los datos.

In [7]:
df_comparendos_electronicos = df_comparendos_electronicos[~((df_comparendos_electronicos['COD_INFRACCION'] == 'C02') & 
                                                             (df_comparendos_electronicos['Tipo Camara'] == 'Fijo'))]

## Preparación de la Serie Temporal Mensual

Se extrae y configura la serie temporal mensual correspondiente a la infracción C32 (no respetar el paso de peatones que cruzan una vía en sitio permitido para ellos o no darles la prelación en las franjas para ello establecidas) a partir del DataFrame de comparendos electrónicos. Para ello, se agrupan los registros por mes y código de infracción, sumando la columna `CANTIDAD_INFRACCIONES` para obtener el volumen real de infracciones por período. La serie resultante se indexa con fechas mensuales, se establece una frecuencia mensual ('MS') y se rellenan los valores faltantes con cero, asegurando una secuencia temporal continua y completa para los modelos predictivos. Se imprime información básica de la serie: número de meses, rango temporal y total de comparendos del período.

In [8]:
df_comparendos_electronicos_copy = df_comparendos_electronicos.copy()
df_comparendos_electronicos_copy['año_mes'] = df_comparendos_electronicos_copy['fecha_comparendo'].dt.to_period('M').astype(str)

def preparar_serie_mensual(df, codigo_infraccion):
    infracciones_por_mes = df.groupby(['año_mes', 'COD_INFRACCION'])['CANTIDAD_INFRACCIONES'].sum().reset_index()
    
    serie = infracciones_por_mes[infracciones_por_mes['COD_INFRACCION'] == codigo_infraccion].copy()
    serie = serie.set_index('año_mes')['CANTIDAD_INFRACCIONES']
    
    serie.index = pd.to_datetime(serie.index)
    serie = serie.asfreq('MS')
    serie = serie.fillna(0)
    
    return serie

serie_c32 = preparar_serie_mensual(df_comparendos_electronicos_copy, 'C32')
print(f"Serie C32: {len(serie_c32)} meses")
print(f"Desde: {serie_c32.index.min().strftime('%Y-%m')} hasta: {serie_c32.index.max().strftime('%Y-%m')}")
print(f"Total comparendos: {serie_c32.sum():,.0f}")

Serie C32: 96 meses
Desde: 2018-01 hasta: 2025-12
Total comparendos: 3,369


## División de la Serie Temporal en Conjuntos de Entrenamiento y Prueba

Se particiona la serie temporal mensual de la infracción C32 (no respetar el paso de peatones que cruzan una vía en sitio permitido para ellos o no darles la prelación en las franjas para ello establecidas) en dos conjuntos: entrenamiento y prueba. El conjunto de entrenamiento comprende los meses desde enero de 2018 hasta diciembre de 2024, abarcando 84 meses, mientras que el conjunto de prueba corresponde al año completo de 2025, con 12 meses. Esta división permite entrenar los modelos predictivos con los datos históricos y evaluar su desempeño pronosticando el período más reciente (2025), cuyos valores reales ya están disponibles en la base de datos para calcular las métricas de error. Se imprime información detallada de ambos conjuntos, incluyendo su rango temporal y número de meses.

In [9]:
train_start = '2018-01-01'
train_end = '2024-12-01'
test_start = '2025-01-01'
test_end = '2025-12-01'

train_c32 = serie_c32[train_start:train_end].copy()
test_c32 = serie_c32[test_start:test_end].copy()

print(f"Train: {train_c32.index.min().strftime('%Y-%m')} a {train_c32.index.max().strftime('%Y-%m')} ({len(train_c32)} meses)")
print(f"Test: {test_c32.index.min().strftime('%Y-%m')} a {test_c32.index.max().strftime('%Y-%m')} ({len(test_c32)} meses)")

Train: 2018-01 a 2024-12 (84 meses)
Test: 2025-01 a 2025-12 (12 meses)


## Generación de Ventanas de Validación Cruzada Temporal

Se implementa una función de validación cruzada específica para series temporales, que respeta el orden cronológico de los datos y evita la fuga de información del futuro. La función `time_series_cv_manual` genera un conjunto de ventanas de entrenamiento y validación, donde el tamaño de cada ventana de validación es de 12 meses (un año completo). Se configuran 4 ventanas de validación, lo que significa que el modelo se evalúa con los últimos 4 años de datos disponibles. Para la infracción C32, se imprimen los rangos temporales de cada fold, mostrando el período de entrenamiento y validación correspondiente. Esta metodología permite evaluar el desempeño predictivo de los modelos en diferentes horizontes temporales de manera robusta y realista.

In [10]:
def time_series_cv_manual(serie, n_ventanas=4, horizonte=12):
    ventanas = []
    
    for i in range(n_ventanas, 0, -1):
        train_end_idx = len(serie) - (i * horizonte)
        train_fold = serie.iloc[:train_end_idx]
        val_fold = serie.iloc[train_end_idx:train_end_idx + horizonte]
        
        ventanas.append((train_fold, val_fold))
        
    return ventanas

ventanas_c32 = time_series_cv_manual(train_c32, n_ventanas=4, horizonte=12)

print("Ventanas de validación generadas:")
for i, (train_fold, val_fold) in enumerate(ventanas_c32, 1):
    print(f"Fold {i}: Train {train_fold.index.min().strftime('%Y-%m')} a {train_fold.index.max().strftime('%Y-%m')} "
          f"({len(train_fold)}m) -> Val {val_fold.index.min().strftime('%Y-%m')} a {val_fold.index.max().strftime('%Y-%m')} "
          f"({len(val_fold)}m)")

Ventanas de validación generadas:
Fold 1: Train 2018-01 a 2020-12 (36m) -> Val 2021-01 a 2021-12 (12m)
Fold 2: Train 2018-01 a 2021-12 (48m) -> Val 2022-01 a 2022-12 (12m)
Fold 3: Train 2018-01 a 2022-12 (60m) -> Val 2023-01 a 2023-12 (12m)
Fold 4: Train 2018-01 a 2023-12 (72m) -> Val 2024-01 a 2024-12 (12m)


## Holt-Winters

In [11]:
def evaluar_holt_winters(train_fold, val_fold, trend=None, seasonal=None,
                         damped_trend=False, seasonal_periods=None):
    """
    Ajusta un modelo de suavizamiento exponencial (Holt-Winters)
    con los componentes especificados.
    """
    modelo = ExponentialSmoothing(
        train_fold,
        trend=trend,
        seasonal=seasonal,
        damped_trend=damped_trend,
        seasonal_periods=seasonal_periods,
        initialization_method='estimated'
    ).fit()
    
    predicciones = modelo.forecast(len(val_fold))
    
    real = val_fold.values
    pred = predicciones.values
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, modelo

In [12]:
configuraciones_hw = [
    {'trend': None, 'seasonal': None, 'damped': False, 'nombre': 'SES (Simple)'},
    {'trend': 'add', 'seasonal': None, 'damped': False, 'nombre': 'Holt lineal'},
    {'trend': 'add', 'seasonal': None, 'damped': True,  'nombre': 'Holt amortiguado'},
    {'trend': None, 'seasonal': 'add', 'damped': False, 'nombre': 'Estacional aditiva'},
    {'trend': 'add', 'seasonal': 'add', 'damped': False, 'nombre': 'Holt‑Winters aditivo'},
    {'trend': 'add', 'seasonal': 'add', 'damped': True,  'nombre': 'HW aditivo amortiguado'},
]

resultados_cv_c32 = []

print("-" * 50)
print("Validación cruzada - Holt-Winters (C32)")
print("-" * 50)

for config in configuraciones_hw:
    nombre = config['nombre']
    print(f"\n--- {nombre} ---")
    
    metricas_folds = {'RMSE': [], 'MAE': [], 'MAPE': [], 'SMAPE': [], 'MSE': []}
    
    for i, (train_fold, val_fold) in enumerate(ventanas_c32, 1):
        pred, met, _ = evaluar_holt_winters(
            train_fold, val_fold,
            trend=config['trend'],
            seasonal=config['seasonal'],
            damped_trend=config['damped'],
            seasonal_periods=12 if config['seasonal'] else None
        )
        metricas_folds['RMSE'].append(met['RMSE'])
        metricas_folds['MAE'].append(met['MAE'])
        metricas_folds['MAPE'].append(met['MAPE'])
        metricas_folds['SMAPE'].append(met['SMAPE'])
        metricas_folds['MSE'].append(met['MSE'])
        
        print(f"  Fold {i}: RMSE={met['RMSE']:.2f}, MAE={met['MAE']:.2f}, "
              f"MAPE={met['MAPE']:.2f}%, SMAPE={met['SMAPE']:.2f}%, MSE={met['MSE']:.2f}")
    
    promedio = {k: np.mean(v) for k, v in metricas_folds.items()}
    resultados_cv_c32.append({
        'modelo': nombre,
        'RMSE': promedio['RMSE'],
        'MAE': promedio['MAE'],
        'MAPE': promedio['MAPE'],
        'SMAPE': promedio['SMAPE'],
        'MSE': promedio['MSE']
    })
    
    print(f"  Promedio -> RMSE={promedio['RMSE']:.2f}, MAE={promedio['MAE']:.2f}, "
          f"MAPE={promedio['MAPE']:.2f}%, SMAPE={promedio['SMAPE']:.2f}%, MSE={promedio['MSE']:.2f}")

df_cv_c32 = pd.DataFrame(resultados_cv_c32)
mejor_idx = df_cv_c32['RMSE'].idxmin()
mejor_nombre = df_cv_c32.loc[mejor_idx, 'modelo']
mejor_config = next(c for c in configuraciones_hw if c['nombre'] == mejor_nombre)

print("\n" + "-" * 50)
print("Mejor modelo Holt‑Winters según RMSE en validación cruzada")
print("-" * 50)
print(f"Modelo seleccionado: {mejor_nombre}")
display(df_cv_c32.round(2).sort_values('RMSE'))

--------------------------------------------------
Validación cruzada - Holt-Winters (C32)
--------------------------------------------------

--- SES (Simple) ---
  Fold 1: RMSE=16.15, MAE=14.94, MAPE=80.64%, SMAPE=51.46%, MSE=260.71
  Fold 2: RMSE=16.32, MAE=11.95, MAPE=32.82%, SMAPE=38.20%, MSE=266.22
  Fold 3: RMSE=21.85, MAE=19.04, MAPE=84.08%, SMAPE=50.27%, MSE=477.22
  Fold 4: RMSE=22.61, MAE=19.39, MAPE=42.23%, SMAPE=50.48%, MSE=511.19
  Promedio -> RMSE=19.23, MAE=16.33, MAPE=59.94%, SMAPE=47.60%, MSE=378.84

--- Holt lineal ---
  Fold 1: RMSE=17.12, MAE=15.87, MAPE=85.47%, SMAPE=53.59%, MSE=293.03
  Fold 2: RMSE=17.56, MAE=12.77, MAPE=33.87%, SMAPE=41.46%, MSE=308.43
  Fold 3: RMSE=23.80, MAE=21.11, MAPE=91.67%, SMAPE=53.82%, MSE=566.33
  Fold 4: RMSE=22.91, MAE=19.74, MAPE=42.85%, SMAPE=51.68%, MSE=524.77
  Promedio -> RMSE=20.35, MAE=17.37, MAPE=63.47%, SMAPE=50.14%, MSE=423.14

--- Holt amortiguado ---
  Fold 1: RMSE=16.82, MAE=15.60, MAPE=84.02%, SMAPE=52.98%, MSE=283.06


,modelo,RMSE,MAE,MAPE,SMAPE,MSE
0,SES (Simple),19.23,16.33,59.94,47.60,378.84
2,Holt amortiguado,19.40,16.51,60.90,47.97,384.81
1,Holt lineal,20.35,17.37,63.47,50.14,423.14
5,HW aditivo amortiguado,29.13,25.13,78.77,110.83,954.67
3,Estacional aditiva,29.19,25.25,79.08,112.44,955.45
4,Holt‑Winters aditivo,30.21,26.35,83.00,115.76,1036.56


In [13]:
print("-" * 50)
print(f"Evaluación final en Test (2025) - {mejor_nombre}")
print("-" * 50)

modelo_final_c32 = ExponentialSmoothing(
    train_c32,
    trend=mejor_config['trend'],
    seasonal=mejor_config['seasonal'],
    damped_trend=mejor_config['damped'],
    seasonal_periods=12 if mejor_config['seasonal'] else None,
    initialization_method='estimated'
).fit()

pred_test_c32 = modelo_final_c32.forecast(len(test_c32))

real_test = test_c32.values
pred_test = pred_test_c32.values

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = real_test - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - SES (Simple)
--------------------------------------------------
RMSE: 18.07
MAE: 16.15
MAPE: 207.37%
SMAPE: 70.24%
MSE: 326.40


In [14]:
meses = ['Enero','Febrero','Marzo','Abril','Mayo','Junio','Julio','Agosto','Septiembre','Octubre','Noviembre','Diciembre']

fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c32.index, y=train_c32.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c32.index, y=test_c32.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c32.index, y=pred_test_c32.values,
    mode='lines+markers', name=f'Predicción {mejor_nombre}',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C32 - {mejor_nombre}: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c32.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_c32.values,
    mode='lines+markers', name=f'Predicción {mejor_nombre}',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c32.values - pred_test_c32.values

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## ARIMA

In [15]:
def evaluar_arima(train_fold, val_fold, order=(1,0,0)):
    """
    Ajusta un modelo ARIMA con el orden (p,d,q) dado.
    Retorna predicciones, métricas y el modelo ajustado.
    """
    modelo = ARIMA(train_fold, order=order)
    modelo_ajustado = modelo.fit()
    
    predicciones = modelo_ajustado.forecast(steps=len(val_fold))
    
    real = val_fold.values
    pred = predicciones.values
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, modelo_ajustado

In [16]:
ordenes_arima = [
    ((1,0,0), 'ARIMA(1,0,0)'),
    ((0,0,1), 'ARIMA(0,0,1)'),
    ((1,0,1), 'ARIMA(1,0,1)'),
    ((2,0,0), 'ARIMA(2,0,0)'),
    ((0,0,2), 'ARIMA(0,0,2)'),
    ((2,0,1), 'ARIMA(2,0,1)'),
    ((1,0,2), 'ARIMA(1,0,2)'),
    # Con d=1
    ((1,1,0), 'ARIMA(1,1,0)'),
    ((0,1,1), 'ARIMA(0,1,1)'),
    ((1,1,1), 'ARIMA(1,1,1)'),
]

resultados_cv_arima = []

print("-" * 50)
print("Validación cruzada - Comparación de modelos ARIMA (C32)")
print("-" * 50)

for orden, nombre in ordenes_arima:
    print(f"\n--- {nombre} ---")
    metricas_folds = {'RMSE': [], 'MAE': [], 'MAPE': [], 'SMAPE': [], 'MSE': []}
    
    for i, (train_fold, val_fold) in enumerate(ventanas_c32, 1):
        predicciones, met, _ = evaluar_arima(train_fold, val_fold, order=orden)
        metricas_folds['RMSE'].append(met['RMSE'])
        metricas_folds['MAE'].append(met['MAE'])
        metricas_folds['MAPE'].append(met['MAPE'])
        metricas_folds['SMAPE'].append(met['SMAPE'])
        metricas_folds['MSE'].append(met['MSE'])
        
        print(f"  Fold {i}: RMSE={met['RMSE']:.2f}, MAE={met['MAE']:.2f}, "
              f"MAPE={met['MAPE']:.2f}%, SMAPE={met['SMAPE']:.2f}%, MSE={met['MSE']:.2f}")
    
    promedio = {k: np.mean(v) for k, v in metricas_folds.items()}
    resultados_cv_arima.append({
        'modelo': nombre,
        'RMSE': promedio['RMSE'],
        'MAE': promedio['MAE'],
        'MAPE': promedio['MAPE'],
        'SMAPE': promedio['SMAPE'],
        'MSE': promedio['MSE']
    })
    
    print(f"  Promedio -> RMSE={promedio['RMSE']:.2f}, MAE={promedio['MAE']:.2f}, "
          f"MAPE={promedio['MAPE']:.2f}%, SMAPE={promedio['SMAPE']:.2f}%, MSE={met['MSE']:.2f}")

df_cv_arima = pd.DataFrame(resultados_cv_arima)
mejor_idx = df_cv_arima['RMSE'].idxmin()
mejor_orden, mejor_nombre = ordenes_arima[mejor_idx]

print("\n" + "-" * 50)
print("Mejor modelo ARIMA según RMSE en validación cruzada")
print("-" * 50)
print(f"Modelo seleccionado: {mejor_nombre}")
display(df_cv_arima.round(2).sort_values('RMSE'))

--------------------------------------------------
Validación cruzada - Comparación de modelos ARIMA (C32)
--------------------------------------------------

--- ARIMA(1,0,0) ---
  Fold 1: RMSE=19.63, MAE=18.36, MAPE=97.89%, SMAPE=59.06%, MSE=385.17
  Fold 2: RMSE=12.77, MAE=10.73, MAPE=43.59%, SMAPE=34.07%, MSE=162.99
  Fold 3: RMSE=14.03, MAE=11.09, MAPE=52.34%, SMAPE=34.38%, MSE=196.70
  Fold 4: RMSE=18.08, MAE=14.57, MAPE=35.07%, SMAPE=35.51%, MSE=326.87
  Promedio -> RMSE=16.12, MAE=13.69, MAPE=57.22%, SMAPE=40.75%, MSE=326.87

--- ARIMA(0,0,1) ---
  Fold 1: RMSE=21.12, MAE=20.06, MAPE=105.42%, SMAPE=62.68%, MSE=446.25
  Fold 2: RMSE=15.12, MAE=13.15, MAPE=56.66%, SMAPE=40.65%, MSE=228.52
  Fold 3: RMSE=11.38, MAE=9.80, MAPE=43.08%, SMAPE=31.85%, MSE=129.59
  Fold 4: RMSE=17.29, MAE=13.55, MAPE=33.16%, SMAPE=32.65%, MSE=298.97
  Promedio -> RMSE=16.23, MAE=14.14, MAPE=59.58%, SMAPE=41.96%, MSE=298.97

--- ARIMA(1,0,1) ---
  Fold 1: RMSE=19.67, MAE=18.37, MAPE=98.06%, SMAPE=59.04%

,modelo,RMSE,MAE,MAPE,SMAPE,MSE
0,"ARIMA(1,0,0)",16.12,13.69,57.22,40.75,267.93
2,"ARIMA(1,0,1)",16.21,13.77,57.61,40.98,270.52
1,"ARIMA(0,0,1)",16.23,14.14,59.58,41.96,275.83
3,"ARIMA(2,0,0)",16.24,13.80,57.76,41.05,271.40
5,"ARIMA(2,0,1)",16.32,13.88,58.09,41.22,274.64
6,"ARIMA(1,0,2)",16.32,13.82,58.32,41.01,273.31
4,"ARIMA(0,0,2)",16.61,14.14,60.68,41.79,282.51
9,"ARIMA(1,1,1)",17.28,14.89,60.79,42.88,311.03
7,"ARIMA(1,1,0)",19.22,16.33,60.06,47.47,378.57
8,"ARIMA(0,1,1)",19.22,16.32,59.90,47.57,378.36


In [17]:
print("-" * 50)
print(f"Evaluación final en Test (2025) - {mejor_nombre}")
print("-" * 50)

modelo_final_arima = ARIMA(train_c32, order=mejor_orden)
modelo_final_arima_ajustado = modelo_final_arima.fit()

pred_test_arima = modelo_final_arima_ajustado.forecast(steps=len(test_c32))

real_test = test_c32.values
pred_test = pred_test_arima.values

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = real_test - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - ARIMA(1,0,0)
--------------------------------------------------
RMSE: 20.73
MAE: 18.80
MAPE: 235.34%
SMAPE: 75.98%
MSE: 429.56


In [18]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c32.index, y=train_c32.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c32.index, y=test_c32.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c32.index, y=pred_test_arima.values,
    mode='lines+markers', name=f'Predicción {mejor_nombre}',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C32 - {mejor_nombre}: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c32.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_arima.values,
    mode='lines+markers', name=f'Predicción {mejor_nombre}',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c32.values - pred_test_arima.values

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## SARIMA

In [19]:
def evaluar_sarima(train_fold, val_fold, order=(1,0,0), seasonal_order=(0,0,0,12)):
    """
    Ajusta un modelo SARIMAX (SARIMA) con los órdenes especificados.
    Retorna predicciones, métricas y el modelo ajustado.
    """
    modelo = SARIMAX(
        train_fold,
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    modelo_ajustado = modelo.fit(disp=False)
    
    predicciones = modelo_ajustado.forecast(steps=len(val_fold))
    
    real = val_fold.values
    pred = predicciones.values
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, modelo_ajustado

In [20]:
configuraciones_sarima = [
    ((1,0,0), (0,0,0,12), 'SARIMA(1,0,0)(0,0,0)[12]'),
    ((1,0,0), (1,0,0,12), 'SARIMA(1,0,0)(1,0,0)[12]'),
    ((1,0,0), (0,0,1,12), 'SARIMA(1,0,0)(0,0,1)[12]'),
    ((1,0,0), (0,1,1,12), 'SARIMA(1,0,0)(0,1,1)[12]'),
    ((0,0,1), (0,0,0,12), 'SARIMA(0,0,1)(0,0,0)[12]'),
    ((0,0,1), (1,0,0,12), 'SARIMA(0,0,1)(1,0,0)[12]'),
    ((0,0,1), (0,0,1,12), 'SARIMA(0,0,1)(0,0,1)[12]'),
    ((0,0,1), (0,1,1,12), 'SARIMA(0,0,1)(0,1,1)[12]'),
    ((1,0,1), (0,0,0,12), 'SARIMA(1,0,1)(0,0,0)[12]'),
    ((1,0,1), (1,0,0,12), 'SARIMA(1,0,1)(1,0,0)[12]'),
    ((1,0,1), (0,0,1,12), 'SARIMA(1,0,1)(0,0,1)[12]'),
    ((1,0,1), (0,1,1,12), 'SARIMA(1,0,1)(0,1,1)[12]'),
    ((1,1,0), (0,0,0,12), 'SARIMA(1,1,0)(0,0,0)[12]'),
    ((0,1,1), (0,0,0,12), 'SARIMA(0,1,1)(0,0,0)[12]'),
]

resultados_cv_sarima = []

print("-" * 50)
print("Validación cruzada - Comparación de modelos SARIMA (C32)")
print("-" * 50)

for order, seasonal_order, nombre in configuraciones_sarima:
    print(f"\n--- {nombre} ---")
    metricas_folds = {'RMSE': [], 'MAE': [], 'MAPE': [], 'SMAPE': [], 'MSE': []}
    
    for i, (train_fold, val_fold) in enumerate(ventanas_c32, 1):
        _, met, _ = evaluar_sarima(train_fold, val_fold, order=order, seasonal_order=seasonal_order)
        metricas_folds['RMSE'].append(met['RMSE'])
        metricas_folds['MAE'].append(met['MAE'])
        metricas_folds['MAPE'].append(met['MAPE'])
        metricas_folds['SMAPE'].append(met['SMAPE'])
        metricas_folds['MSE'].append(met['MSE'])
        
        print(f"  Fold {i}: RMSE={met['RMSE']:.2f}, MAE={met['MAE']:.2f}, "
              f"MAPE={met['MAPE']:.2f}%, SMAPE={met['SMAPE']:.2f}%, MSE={met['MSE']:.2f}")
    
    promedio = {k: np.mean(v) for k, v in metricas_folds.items()}
    resultados_cv_sarima.append({
        'modelo': nombre,
        'RMSE': promedio['RMSE'],
        'MAE': promedio['MAE'],
        'MAPE': promedio['MAPE'],
        'SMAPE': promedio['SMAPE'],
        'MSE': promedio['MSE']
    })
    
    print(f"  Promedio -> RMSE={promedio['RMSE']:.2f}, MAE={promedio['MAE']:.2f}, "
          f"MAPE={promedio['MAPE']:.2f}%, SMAPE={promedio['SMAPE']:.2f}%, MSE={promedio['MSE']:.2f}")

df_cv_sarima = pd.DataFrame(resultados_cv_sarima)
mejor_idx = df_cv_sarima['RMSE'].idxmin()
mejor_nombre = df_cv_sarima.loc[mejor_idx, 'modelo']
mejor_order, mejor_seasonal, _ = next((o, s, n) for o, s, n in configuraciones_sarima if n == mejor_nombre)

print("\n" + "-" * 50)
print("Mejor modelo SARIMA según RMSE en validación cruzada")
print("-" * 50)
print(f"Modelo seleccionado: {mejor_nombre}")
display(df_cv_sarima.round(2).sort_values('RMSE'))

--------------------------------------------------
Validación cruzada - Comparación de modelos SARIMA (C32)
--------------------------------------------------

--- SARIMA(1,0,0)(0,0,0)[12] ---
  Fold 1: RMSE=6.04, MAE=4.75, MAPE=21.37%, SMAPE=23.33%, MSE=36.47
  Fold 2: RMSE=24.61, MAE=18.19, MAPE=45.92%, SMAPE=69.61%, MSE=605.81
  Fold 3: RMSE=13.02, MAE=10.47, MAPE=40.98%, SMAPE=34.78%, MSE=169.42
  Fold 4: RMSE=32.46, MAE=29.16, MAPE=60.87%, SMAPE=92.43%, MSE=1053.86
  Promedio -> RMSE=19.03, MAE=15.64, MAPE=42.29%, SMAPE=55.04%, MSE=466.39

--- SARIMA(1,0,0)(1,0,0)[12] ---
  Fold 1: RMSE=8.85, MAE=7.01, MAPE=28.95%, SMAPE=37.03%, MSE=78.33
  Fold 2: RMSE=26.90, MAE=20.84, MAPE=55.52%, SMAPE=87.66%, MSE=723.67
  Fold 3: RMSE=15.15, MAE=12.13, MAPE=42.94%, SMAPE=42.12%, MSE=229.57
  Fold 4: RMSE=34.41, MAE=31.60, MAPE=67.61%, SMAPE=106.05%, MSE=1184.09
  Promedio -> RMSE=21.33, MAE=17.90, MAPE=48.75%, SMAPE=68.22%, MSE=553.92

--- SARIMA(1,0,0)(0,0,1)[12] ---
  Fold 1: RMSE=10.62, MA

,modelo,RMSE,MAE,MAPE,SMAPE,MSE
8,"SARIMA(1,0,1)(0,0,0)[12]",18.08,14.71,40.48,49.98,420.95
3,"SARIMA(1,0,0)(0,1,1)[12]",18.10,14.80,55.08,46.85,329.01
10,"SARIMA(1,0,1)(0,0,1)[12]",18.50,15.22,41.22,52.35,440.54
0,"SARIMA(1,0,0)(0,0,0)[12]",19.03,15.64,42.29,55.04,466.39
12,"SARIMA(1,1,0)(0,0,0)[12]",19.23,16.34,60.10,47.50,378.98
13,"SARIMA(0,1,1)(0,0,0)[12]",19.24,16.34,59.99,47.64,379.32
7,"SARIMA(0,0,1)(0,1,1)[12]",19.35,16.52,60.32,49.15,381.26
9,"SARIMA(1,0,1)(1,0,0)[12]",19.82,16.80,46.55,62.62,504.17
11,"SARIMA(1,0,1)(0,1,1)[12]",19.84,15.92,55.75,54.14,398.72
5,"SARIMA(0,0,1)(1,0,0)[12]",20.99,17.67,48.68,67.33,491.00


In [21]:
print("-" * 50)
print(f"Evaluación final en Test (2025) - {mejor_nombre}")
print("-" * 50)

modelo_final_sarima = SARIMAX(
    train_c32,
    order=mejor_order,
    seasonal_order=mejor_seasonal,
    enforce_stationarity=False,
    enforce_invertibility=False
)
modelo_final_sarima_ajustado = modelo_final_sarima.fit(disp=False)

pred_test_sarima = modelo_final_sarima_ajustado.forecast(steps=len(test_c32))

real_test = test_c32.values
pred_test = pred_test_sarima.values

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = real_test - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - SARIMA(1,0,1)(0,0,0)[12]
--------------------------------------------------
RMSE: 7.87
MAE: 6.61
MAPE: 88.69%
SMAPE: 43.05%
MSE: 61.98


In [22]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c32.index, y=train_c32.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c32.index, y=test_c32.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c32.index, y=pred_test_sarima.values,
    mode='lines+markers', name=f'Predicción {mejor_nombre}',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C32 - {mejor_nombre}: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c32.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_sarima.values,
    mode='lines+markers', name=f'Predicción {mejor_nombre}',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c32.values - pred_test_sarima.values

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## Dynamic Optimized Theta

In [23]:
def evaluar_theta(train_fold, val_fold, theta=2.0, period=12):
    """
    Ajusta un modelo Theta con el parámetro theta dado.
    Procedimiento:
      1. Descomponer la serie con STL (period=12) para obtener estacionalidad.
      2. Serie desestacionalizada = serie - estacionalidad.
      3. Ajustar regresión lineal a la serie desestacionalizada (tendencia global).
      4. Ajustar SES a la serie desestacionalizada.
      5. Combinar pronósticos: Yhat = SES_forecast + theta*(trend_forecast - SES_forecast).
      6. Sumar estacionalidad (último año).
    Retorna predicciones, métricas y un diccionario con información del modelo.
    """
    if len(train_fold) < 2 * period:
        ses = ExponentialSmoothing(train_fold, trend=None, seasonal=None,
                                   initialization_method='estimated').fit()
        pred = ses.forecast(len(val_fold))
        real = val_fold.values
        pred = pred.values
        mse = mean_squared_error(real, pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(real, pred)
        mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
        smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
        return pred, {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'SMAPE': smape}, {'theta': theta, 'method': 'SES'}
    
    stl = STL(train_fold, period=period, robust=True).fit()
    seasonal = stl.seasonal
    deseasonalized = train_fold - seasonal

    x = np.arange(len(deseasonalized))
    slope, intercept, _, _, _ = sp_stats.linregress(x, deseasonalized.values)
    trend_line = slope * x + intercept

    ses_model = ExponentialSmoothing(deseasonalized, trend=None, seasonal=None,
                                     initialization_method='estimated').fit()
    
    h = len(val_fold)
    ses_forecast = ses_model.forecast(h)
    last_idx = len(deseasonalized) - 1
    trend_extrapolated = slope * np.arange(last_idx + 1, last_idx + 1 + h) + intercept

    theta_forecast = ses_forecast.values + theta * (trend_extrapolated - ses_forecast.values)

    last_year_seasonal = seasonal.iloc[-period:].values
    repeats = int(np.ceil(h / period))
    seasonal_pattern = np.tile(last_year_seasonal, repeats)[:h]
    final_forecast = theta_forecast + seasonal_pattern

    real = val_fold.values
    pred = final_forecast

    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'SMAPE': smape}
    modelo_info = {'theta': theta, 'slope': slope, 'intercept': intercept}
    
    return final_forecast, metricas, modelo_info

In [24]:
thetas = np.arange(0.0, 4.0 + 0.25, 0.25)

resultados_cv_theta = []

print("-" * 50)
print("Validación cruzada - Optimización de Theta (C32)")
print("-" * 50)

for theta in thetas:
    metricas_folds = {'RMSE': [], 'MAE': [], 'MAPE': [], 'SMAPE': [], 'MSE': []}
    
    for i, (train_fold, val_fold) in enumerate(ventanas_c32, 1):
        _, met, _ = evaluar_theta(train_fold, val_fold, theta=theta, period=12)
        metricas_folds['RMSE'].append(met['RMSE'])
        metricas_folds['MAE'].append(met['MAE'])
        metricas_folds['MAPE'].append(met['MAPE'])
        metricas_folds['SMAPE'].append(met['SMAPE'])
        metricas_folds['MSE'].append(met['MSE'])
    
    promedio = {k: np.mean(v) for k, v in metricas_folds.items()}
    resultados_cv_theta.append({
        'theta': theta,
        'RMSE': promedio['RMSE'],
        'MAE': promedio['MAE'],
        'MAPE': promedio['MAPE'],
        'SMAPE': promedio['SMAPE'],
        'MSE': promedio['MSE']
    })

df_cv_theta = pd.DataFrame(resultados_cv_theta)
mejor_idx = df_cv_theta['RMSE'].idxmin()
mejor_theta = df_cv_theta.loc[mejor_idx, 'theta']
mejor_rmse_cv = df_cv_theta.loc[mejor_idx, 'RMSE']

print("\n--- Resultados de optimización de theta ---")
print(f"Mejor theta: {mejor_theta:.2f} (RMSE CV promedio = {mejor_rmse_cv:.2f})")
print("\nTop 5 configuraciones:")
display(df_cv_theta.sort_values('RMSE').head())

print("\n" + "-" * 50)
print(f"Validación cruzada - Theta dinámico óptimo (θ={mejor_theta:.2f})")
print("-" * 50)

metricas_opt_folds = {'RMSE': [], 'MAE': [], 'MAPE': [], 'SMAPE': [], 'MSE': []}
for i, (train_fold, val_fold) in enumerate(ventanas_c32, 1):
    _, met, _ = evaluar_theta(train_fold, val_fold, theta=mejor_theta, period=12)
    metricas_opt_folds['RMSE'].append(met['RMSE'])
    metricas_opt_folds['MAE'].append(met['MAE'])
    metricas_opt_folds['MAPE'].append(met['MAPE'])
    metricas_opt_folds['SMAPE'].append(met['SMAPE'])
    metricas_opt_folds['MSE'].append(met['MSE'])
    print(f"  Fold {i}: RMSE={met['RMSE']:.2f}, MAE={met['MAE']:.2f}, "
          f"MAPE={met['MAPE']:.2f}%, SMAPE={met['SMAPE']:.2f}%, MSE={met['MSE']:.2f}")

prom_opt = {k: np.mean(v) for k, v in metricas_opt_folds.items()}
print(f"  Promedio -> RMSE={prom_opt['RMSE']:.2f}, MAE={prom_opt['MAE']:.2f}, "
      f"MAPE={prom_opt['MAPE']:.2f}%, SMAPE={prom_opt['SMAPE']:.2f}%, MSE={promedio['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - Optimización de Theta (C32)
--------------------------------------------------

--- Resultados de optimización de theta ---
Mejor theta: 0.25 (RMSE CV promedio = 19.38)

Top 5 configuraciones:


,theta,RMSE,MAE,MAPE,SMAPE,MSE
1,0.25,19.378681,16.073016,57.622781,52.655346,380.169678
0,0.00,19.516415,16.095518,57.217780,55.962050,385.305057
2,0.50,19.602267,16.224989,58.684534,50.661356,394.526123
3,0.75,20.175755,16.750637,61.261396,50.507509,428.374390
4,1.00,21.079436,17.762538,65.655802,52.219356,481.714479



--------------------------------------------------
Validación cruzada - Theta dinámico óptimo (θ=0.25)
--------------------------------------------------
  Fold 1: RMSE=20.54, MAE=17.26, MAPE=95.79%, SMAPE=59.84%, MSE=421.93
  Fold 2: RMSE=20.85, MAE=16.28, MAPE=45.19%, SMAPE=62.76%, MSE=434.54
  Fold 3: RMSE=15.66, MAE=12.82, MAPE=51.37%, SMAPE=38.49%, MSE=245.15
  Fold 4: RMSE=20.47, MAE=17.93, MAPE=38.13%, SMAPE=49.54%, MSE=419.06
  Promedio -> RMSE=19.38, MAE=16.07, MAPE=57.62%, SMAPE=52.66%, MSE=2642.16


In [25]:
print("-" * 50)
print(f"Evaluación final en Test (2025) - Theta optimizado (θ={mejor_theta:.2f})")
print("-" * 50)

pred_test_theta, met_train, modelo_info = evaluar_theta(train_c32, test_c32, theta=mejor_theta, period=12)

real_test = test_c32.values
pred_test = pred_test_theta

mse_test = met_train['MSE']
rmse_test = met_train['RMSE']
mae_test = met_train['MAE']
mape_test = met_train['MAPE']
smape_test = met_train['SMAPE']

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c32.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - Theta optimizado (θ=0.25)
--------------------------------------------------
RMSE: 31.34
MAE: 29.02
MAPE: 265.39%
SMAPE: 94.72%
MSE: 981.92


In [26]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c32.index, y=train_c32.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c32.index, y=test_c32.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c32.index, y=pred_test_theta,
    mode='lines+markers', name=f'Predicción Theta (θ={mejor_theta:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C32 - Theta Dinámico Optimizado (θ={mejor_theta:.2f}): Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c32.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_theta,
    mode='lines+markers', name=f'Predicción Theta (θ={mejor_theta:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c32.values - pred_test_theta

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## Ridge

In [27]:

def crear_dataset_supervisado_ridge(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas.
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_ridge(train_fold, val_fold, n_input=12, alphas=None):
    """
    Ajusta RidgeCV con características determinísticas.
    Retorna predicciones, métricas, modelo y scaler.
    """
    if alphas is None:
        alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
    
    X_train, y_train = crear_dataset_supervisado_ridge(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    tscv = TimeSeriesSplit(n_splits=3)
    modelo = RidgeCV(alphas=alphas, cv=tscv)
    modelo.fit(X_train_scaled, y_train)
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, modelo, scaler

In [28]:
resultados_cv_ridge = []

print("-" * 50)
print("Validación cruzada - Ridge Regression (C32)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_c32, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_ridge(train_fold, val_fold)
    
    resultados_cv_ridge.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'alpha': modelo.alpha_,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Alpha seleccionado: {modelo.alpha_:.2f}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_ridge = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_ridge]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_ridge]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_ridge]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_ridge]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_ridge])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_ridge['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_ridge['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_ridge['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_ridge['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_ridge['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - Ridge Regression (C32)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Alpha seleccionado: 100.00
RMSE: 21.77
MAE: 20.39
MAPE: 100.34%
SMAPE: 62.10%
MSE: 473.79

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Alpha seleccionado: 100.00
RMSE: 17.88
MAE: 15.39
MAPE: 57.77%
SMAPE: 47.14%
MSE: 319.80

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Alpha seleccionado: 100.00
RMSE: 12.37
MAE: 10.35
MAPE: 39.13%
SMAPE: 33.77%
MSE: 152.92

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Alpha seleccionado: 1000.00
RMSE: 18.50
MAE: 14.52
MAPE: 33.12%
SMAPE: 35.36%
MSE: 342.39

--------------------------------------------------
Métricas promedio en validación cruzada
--------------------------------------------------
RMSE: 17.63
MAE: 15.16
MAPE: 57.59%
SMAPE: 44.59%
MSE: 322.23


In [29]:
print("-" * 50)
print("Evaluación final en Test (2025) - Ridge Regression")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_ridge(
    train_c32.values, train_c32.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

tscv = TimeSeriesSplit(n_splits=3)
alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
modelo_final_ridge = RidgeCV(alphas=alphas, cv=tscv)
modelo_final_ridge.fit(X_train_full_scaled, y_train_full)
print(f"Alpha seleccionado: {modelo_final_ridge.alpha_:.2f}")

x_lags_pred_test = train_c32.values[-12:]
fecha_fin_pred_test = train_c32.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_ridge = modelo_final_ridge.predict(X_pred_test_scaled).flatten()

real_test = test_c32.values
pred_test = pred_test_ridge

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c32.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - Ridge Regression
--------------------------------------------------
Alpha seleccionado: 1000.00
RMSE: 18.82
MAE: 17.15
MAPE: 214.60%
SMAPE: 72.79%
MSE: 354.37


In [30]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c32.index, y=train_c32.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c32.index, y=test_c32.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c32.index, y=pred_test_ridge,
    mode='lines+markers', name=f'Predicción Ridge (α={modelo_final_ridge.alpha_:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C32 - Ridge Regression: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c32.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_ridge,
    mode='lines+markers', name=f'Predicción Ridge (α={modelo_final_ridge.alpha_:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c32.values - pred_test_ridge

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## Lasso

In [31]:
def crear_dataset_supervisado_lasso(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas.
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_lasso(train_fold, val_fold, n_input=12, alphas=None):
    """
    Ajusta MultiTaskLassoCV con características determinísticas.
    Retorna predicciones, métricas, modelo y scaler.
    """
    if alphas is None:
        alphas = np.logspace(-4, 4, 50)
    
    X_train, y_train = crear_dataset_supervisado_lasso(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    tscv = TimeSeriesSplit(n_splits=3)
    modelo = MultiTaskLassoCV(alphas=alphas, cv=tscv, max_iter=20000, random_state=42)
    modelo.fit(X_train_scaled, y_train)
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, modelo, scaler

In [32]:
resultados_cv_lasso = []

print("-" * 50)
print("Validación cruzada - Lasso Regression (C32)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_c32, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_lasso(train_fold, val_fold)
    
    resultados_cv_lasso.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'alpha': modelo.alpha_,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Alpha seleccionado: {modelo.alpha_:.2f}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_lasso = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_lasso]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_lasso]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_lasso]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_lasso]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_lasso])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_lasso['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_lasso['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_lasso['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_lasso['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_lasso['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - Lasso Regression (C32)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Alpha seleccionado: 7.91
RMSE: 16.10
MAE: 11.92
MAPE: 67.29%
SMAPE: 47.95%
MSE: 259.06

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Alpha seleccionado: 24.42
RMSE: 16.60
MAE: 13.82
MAPE: 50.14%
SMAPE: 43.53%
MSE: 275.44

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Alpha seleccionado: 16.77
RMSE: 12.55
MAE: 10.55
MAPE: 35.78%
SMAPE: 35.08%
MSE: 157.54

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Alpha seleccionado: 24.42
RMSE: 18.42
MAE: 14.52
MAPE: 33.29%
SMAPE: 35.40%
MSE: 339.45

--------------------------------------------------
Métricas promedio en validación cruzada
--------------------------------------------------
RMSE: 15.92
MAE: 12.70
MAPE: 46.63%
SMAPE: 40.49%
MSE: 257.87


In [33]:
print("-" * 50)
print("Evaluación final en Test (2025) - Lasso Regression")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_lasso(
    train_c32.values, train_c32.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

tscv = TimeSeriesSplit(n_splits=3)
alphas = np.logspace(-4, 4, 50)
modelo_final_lasso = MultiTaskLassoCV(alphas=alphas, cv=tscv, max_iter=20000, random_state=42)
modelo_final_lasso.fit(X_train_full_scaled, y_train_full)
print(f"Alpha seleccionado: {modelo_final_lasso.alpha_:.2f}")

x_lags_pred_test = train_c32.values[-12:]
fecha_fin_pred_test = train_c32.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_lasso = modelo_final_lasso.predict(X_pred_test_scaled).flatten()

real_test = test_c32.values
pred_test = pred_test_lasso

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c32.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - Lasso Regression
--------------------------------------------------
Alpha seleccionado: 24.42
RMSE: 18.85
MAE: 17.28
MAPE: 213.82%
SMAPE: 73.22%
MSE: 355.25


In [34]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c32.index, y=train_c32.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c32.index, y=test_c32.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c32.index, y=pred_test_lasso,
    mode='lines+markers', name=f'Predicción Lasso (α={modelo_final_lasso.alpha_:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C32 - Lasso Regression: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c32.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_lasso,
    mode='lines+markers', name=f'Predicción Lasso (α={modelo_final_lasso.alpha_:.2f})',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c32.values - pred_test_lasso

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## Random Forest

In [35]:
def crear_dataset_supervisado_rf(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas.
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_random_forest(train_fold, val_fold, n_input=12, param_grid=None):
    """
    Ajusta Random Forest multi-output con búsqueda de hiperparámetros.
    Retorna predicciones, métricas, mejor modelo y scaler.
    """
    if param_grid is None:
        param_grid = {
            'estimator__n_estimators': [100, 200, 300],
            'estimator__max_depth': [5, 10, 15, None],
            'estimator__min_samples_split': [2, 5, 10]
        }
    
    X_train, y_train = crear_dataset_supervisado_rf(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    rf = RandomForestRegressor(random_state=42)
    multi_rf = MultiOutputRegressor(rf)
    
    tscv = TimeSeriesSplit(n_splits=3)
    grid = GridSearchCV(multi_rf, param_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(X_train_scaled, y_train)
    
    mejor_modelo = grid.best_estimator_
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = mejor_modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, mejor_modelo, scaler

In [36]:
resultados_cv_rf = []

print("-" * 50)
print("Validación cruzada - Random Forest (C32)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_c32, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_random_forest(train_fold, val_fold)
    
    best_params = modelo.get_params()
    n_estimators = best_params['estimator__n_estimators']
    max_depth = best_params['estimator__max_depth']
    min_samples_split = best_params['estimator__min_samples_split']
    
    resultados_cv_rf.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'min_samples_split': min_samples_split,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Mejores parámetros: n_estimators={n_estimators}, max_depth={max_depth}, min_samples_split={min_samples_split}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_rf = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_rf]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_rf]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_rf]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_rf]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_rf])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_rf['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_rf['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_rf['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_rf['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_rf['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - Random Forest (C32)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Mejores parámetros: n_estimators=100, max_depth=10, min_samples_split=2
RMSE: 23.60
MAE: 21.83
MAPE: 109.46%
SMAPE: 64.55%
MSE: 556.82

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Mejores parámetros: n_estimators=300, max_depth=10, min_samples_split=5
RMSE: 20.48
MAE: 17.48
MAPE: 64.26%
SMAPE: 53.84%
MSE: 419.35

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Mejores parámetros: n_estimators=100, max_depth=5, min_samples_split=5
RMSE: 14.73
MAE: 13.02
MAPE: 47.39%
SMAPE: 42.90%
MSE: 216.92

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Mejores parámetros: n_estimators=100, max_depth=5, min_samples_split=10
RMSE: 19.25
MAE: 16.67
MAPE: 36.65%
SMAPE: 42.62%
MSE: 370.60

--------------------------------------------------
Métricas prom

In [37]:
print("-" * 50)
print("Evaluación final en Test (2025) - Random Forest")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_rf(
    train_c32.values, train_c32.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

param_grid_final = {
    'estimator__n_estimators': [100, 200, 300, 400],
    'estimator__max_depth': [5, 10, 15, None],
    'estimator__min_samples_split': [2, 5, 10]
}

rf_final = RandomForestRegressor(random_state=42)
multi_rf_final = MultiOutputRegressor(rf_final)
tscv = TimeSeriesSplit(n_splits=3)
grid_final = GridSearchCV(multi_rf_final, param_grid_final, cv=tscv, 
                          scoring='neg_mean_squared_error', n_jobs=-1)
grid_final.fit(X_train_full_scaled, y_train_full)

modelo_final_rf = grid_final.best_estimator_
best_params_final = grid_final.best_params_
print(f"Mejores parámetros: {best_params_final}")

x_lags_pred_test = train_c32.values[-12:]
fecha_fin_pred_test = train_c32.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_rf = modelo_final_rf.predict(X_pred_test_scaled).flatten()

real_test = test_c32.values
pred_test = pred_test_rf

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c32.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - Random Forest
--------------------------------------------------
Mejores parámetros: {'estimator__max_depth': 5, 'estimator__min_samples_split': 5, 'estimator__n_estimators': 400}
RMSE: 26.05
MAE: 24.08
MAPE: 286.61%
SMAPE: 85.71%
MSE: 678.79


In [38]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c32.index, y=train_c32.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c32.index, y=test_c32.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c32.index, y=pred_test_rf,
    mode='lines+markers', name='Predicción Random Forest',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C32 - Random Forest: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c32.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_rf,
    mode='lines+markers', name='Predicción Random Forest',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c32.values - pred_test_rf

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## XGBoost

In [39]:
def crear_dataset_supervisado_xgb(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas.
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_xgboost(train_fold, val_fold, n_input=12, param_grid=None):
    """
    Ajusta XGBoost multi-output con búsqueda de hiperparámetros.
    Retorna predicciones, métricas, mejor modelo y scaler.
    """
    if param_grid is None:
        param_grid = {
            'estimator__n_estimators': [100, 200, 300],
            'estimator__max_depth': [3, 5, 7],
            'estimator__learning_rate': [0.01, 0.05, 0.1],
            'estimator__subsample': [0.8, 1.0],
            'estimator__colsample_bytree': [0.8, 1.0]
        }
    
    X_train, y_train = crear_dataset_supervisado_xgb(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    xgb = XGBRegressor(random_state=42, objective='reg:squarederror', verbosity=0)
    multi_xgb = MultiOutputRegressor(xgb)
    
    tscv = TimeSeriesSplit(n_splits=3)
    grid = GridSearchCV(multi_xgb, param_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(X_train_scaled, y_train)
    
    mejor_modelo = grid.best_estimator_
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = mejor_modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, mejor_modelo, scaler

In [40]:
resultados_cv_xgb = []

print("-" * 50)
print("Validación cruzada - XGBoost (C32)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_c32, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_xgboost(train_fold, val_fold)
    
    best_params = modelo.get_params()
    n_estimators = best_params['estimator__n_estimators']
    max_depth = best_params['estimator__max_depth']
    learning_rate = best_params['estimator__learning_rate']
    subsample = best_params['estimator__subsample']
    colsample_bytree = best_params['estimator__colsample_bytree']
    
    resultados_cv_xgb.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'learning_rate': learning_rate,
        'subsample': subsample,
        'colsample_bytree': colsample_bytree,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Mejores parámetros: n_estimators={n_estimators}, max_depth={max_depth}, learning_rate={learning_rate}, subsample={subsample}, colsample_bytree={colsample_bytree}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_xgb = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_xgb]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_xgb]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_xgb]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_xgb]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_xgb])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_xgb['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_xgb['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_xgb['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_xgb['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_xgb['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - XGBoost (C32)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Mejores parámetros: n_estimators=100, max_depth=5, learning_rate=0.01, subsample=1.0, colsample_bytree=1.0
RMSE: 29.03
MAE: 24.61
MAPE: 115.64%
SMAPE: 66.72%
MSE: 842.47

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Mejores parámetros: n_estimators=100, max_depth=7, learning_rate=0.01, subsample=0.8, colsample_bytree=0.8
RMSE: 21.46
MAE: 18.62
MAPE: 73.26%
SMAPE: 53.80%
MSE: 460.56

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Mejores parámetros: n_estimators=100, max_depth=3, learning_rate=0.01, subsample=1.0, colsample_bytree=0.8
RMSE: 13.45
MAE: 11.52
MAPE: 43.16%
SMAPE: 37.79%
MSE: 180.93

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Mejores parámetros: n_estimators=100, max_depth=3, learning_rate=0.01, subsample=0.8, colsample_byt

In [41]:
print("-" * 50)
print("Evaluación final en Test (2025) - XGBoost")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_xgb(
    train_c32.values, train_c32.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

param_grid_final = {
    'estimator__n_estimators': [100, 200, 300, 400],
    'estimator__max_depth': [3, 5, 7, 9],
    'estimator__learning_rate': [0.01, 0.05, 0.1],
    'estimator__subsample': [0.8, 1.0],
    'estimator__colsample_bytree': [0.8, 1.0]
}

xgb_final = XGBRegressor(random_state=42, objective='reg:squarederror', verbosity=0)
multi_xgb_final = MultiOutputRegressor(xgb_final)
tscv = TimeSeriesSplit(n_splits=3)
grid_final = GridSearchCV(multi_xgb_final, param_grid_final, cv=tscv, 
                          scoring='neg_mean_squared_error', n_jobs=-1)
grid_final.fit(X_train_full_scaled, y_train_full)

modelo_final_xgb = grid_final.best_estimator_
best_params_final = grid_final.best_params_
print(f"Mejores parámetros: {best_params_final}")

x_lags_pred_test = train_c32.values[-12:]
fecha_fin_pred_test = train_c32.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_xgb = modelo_final_xgb.predict(X_pred_test_scaled).flatten()

real_test = test_c32.values
pred_test = pred_test_xgb

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c32.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - XGBoost
--------------------------------------------------
Mejores parámetros: {'estimator__colsample_bytree': 0.8, 'estimator__learning_rate': 0.01, 'estimator__max_depth': 5, 'estimator__n_estimators': 100, 'estimator__subsample': 1.0}
RMSE: 21.97
MAE: 20.40
MAPE: 240.94%
SMAPE: 79.64%
MSE: 482.73


In [42]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c32.index, y=train_c32.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c32.index, y=test_c32.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c32.index, y=pred_test_xgb,
    mode='lines+markers', name='Predicción XGBoost',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C32 - XGBoost: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c32.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_xgb,
    mode='lines+markers', name='Predicción XGBoost',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c32.values - pred_test_xgb

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## LightGBM

In [43]:
def crear_dataset_supervisado_lgb(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas.
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_lightgbm(train_fold, val_fold, n_input=12, param_grid=None):
    """
    Ajusta LightGBM multi-output con búsqueda de hiperparámetros.
    Retorna predicciones, métricas, mejor modelo y scaler.
    """
    if param_grid is None:
        param_grid = {
            'estimator__n_estimators': [100, 200, 300],
            'estimator__max_depth': [3, 5, 7, -1],
            'estimator__learning_rate': [0.01, 0.05, 0.1],
            'estimator__subsample': [0.8, 1.0],
            'estimator__colsample_bytree': [0.8, 1.0]
        }
    
    X_train, y_train = crear_dataset_supervisado_lgb(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    lgb = LGBMRegressor(random_state=42, verbosity=-1)
    multi_lgb = MultiOutputRegressor(lgb)
    
    tscv = TimeSeriesSplit(n_splits=3)
    grid = GridSearchCV(multi_lgb, param_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(X_train_scaled, y_train)
    
    mejor_modelo = grid.best_estimator_
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = mejor_modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, mejor_modelo, scaler

In [44]:
resultados_cv_lgb = []

print("-" * 50)
print("Validación cruzada - LightGBM (C32)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_c32, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_lightgbm(train_fold, val_fold)
    
    best_params = modelo.get_params()
    n_estimators = best_params['estimator__n_estimators']
    max_depth = best_params['estimator__max_depth']
    learning_rate = best_params['estimator__learning_rate']
    subsample = best_params['estimator__subsample']
    colsample_bytree = best_params['estimator__colsample_bytree']
    
    resultados_cv_lgb.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'learning_rate': learning_rate,
        'subsample': subsample,
        'colsample_bytree': colsample_bytree,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Mejores parámetros: n_estimators={n_estimators}, max_depth={max_depth}, learning_rate={learning_rate}, subsample={subsample}, colsample_bytree={colsample_bytree}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_lgb = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_lgb]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_lgb]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_lgb]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_lgb]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_lgb])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_lgb['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_lgb['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_lgb['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_lgb['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_lgb['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - LightGBM (C32)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Mejores parámetros: n_estimators=100, max_depth=3, learning_rate=0.01, subsample=0.8, colsample_bytree=0.8
RMSE: 23.54
MAE: 22.37
MAPE: 108.34%
SMAPE: 66.26%
MSE: 554.35

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Mejores parámetros: n_estimators=100, max_depth=3, learning_rate=0.01, subsample=0.8, colsample_bytree=0.8
RMSE: 19.00
MAE: 16.30
MAPE: 66.83%
SMAPE: 47.90%
MSE: 360.90

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Mejores parámetros: n_estimators=100, max_depth=3, learning_rate=0.01, subsample=0.8, colsample_bytree=0.8
RMSE: 11.55
MAE: 9.27
MAPE: 39.59%
SMAPE: 30.06%
MSE: 133.33

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Mejores parámetros: n_estimators=100, max_depth=3, learning_rate=0.01, subsample=0.8, colsample_byt

In [45]:
print("-" * 50)
print("Evaluación final en Test (2025) - LightGBM")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_lgb(
    train_c32.values, train_c32.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

param_grid_final = {
    'estimator__n_estimators': [100, 200, 300, 400],
    'estimator__max_depth': [3, 5, 7, -1],
    'estimator__learning_rate': [0.01, 0.05, 0.1],
    'estimator__subsample': [0.8, 1.0],
    'estimator__colsample_bytree': [0.8, 1.0]
}

lgb_final = LGBMRegressor(random_state=42, verbosity=-1)
multi_lgb_final = MultiOutputRegressor(lgb_final)
tscv = TimeSeriesSplit(n_splits=3)
grid_final = GridSearchCV(multi_lgb_final, param_grid_final, cv=tscv, 
                          scoring='neg_mean_squared_error', n_jobs=-1)
grid_final.fit(X_train_full_scaled, y_train_full)

modelo_final_lgb = grid_final.best_estimator_
best_params_final = grid_final.best_params_
print(f"Mejores parámetros: {best_params_final}")

x_lags_pred_test = train_c32.values[-12:]
fecha_fin_pred_test = train_c32.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_lgb = modelo_final_lgb.predict(X_pred_test_scaled).flatten()

real_test = test_c32.values
pred_test = pred_test_lgb

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c32.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - LightGBM
--------------------------------------------------
Mejores parámetros: {'estimator__colsample_bytree': 0.8, 'estimator__learning_rate': 0.01, 'estimator__max_depth': 3, 'estimator__n_estimators': 100, 'estimator__subsample': 0.8}
RMSE: 18.59
MAE: 16.30
MAPE: 218.21%
SMAPE: 69.89%
MSE: 345.44


In [46]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c32.index, y=train_c32.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c32.index, y=test_c32.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c32.index, y=pred_test_lgb,
    mode='lines+markers', name='Predicción LightGBM',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C32 - LightGBM: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c32.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_lgb,
    mode='lines+markers', name='Predicción LightGBM',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c32.values - pred_test_lgb

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## KNN

In [47]:
def crear_dataset_supervisado_knn(serie_valores, fechas, n_input=12, n_output=12):
    """
    Crea X con rezagos y características determinísticas.
    Retorna X, y.
    """
    X, y = [], []
    for i in range(len(serie_valores) - n_input - n_output + 1):
        x_lags = serie_valores[i:i + n_input]
        fecha_fin_input = fechas[i + n_input - 1]
        tendencia = (i + n_input) / len(serie_valores)
        mes = fecha_fin_input.month
        mes_sin = np.sin(2 * np.pi * mes / 12)
        mes_cos = np.cos(2 * np.pi * mes / 12)
        fecha_primer_pred = fechas[i + n_input]
        covid = 1 if (fecha_primer_pred >= pd.Timestamp('2020-03-01') and 
                      fecha_primer_pred <= pd.Timestamp('2020-12-01')) else 0
        x_full = np.concatenate([x_lags, [tendencia, mes_sin, mes_cos, covid]])
        X.append(x_full)
        y.append(serie_valores[i + n_input:i + n_input + n_output])
    return np.array(X), np.array(y)

def evaluar_knn(train_fold, val_fold, n_input=12, param_grid=None):
    """
    Ajusta KNN multi‑output con búsqueda de hiperparámetros.
    Retorna predicciones, métricas, mejor modelo y scaler.
    """
    if param_grid is None:
        param_grid = {
            'estimator__n_neighbors': [2, 3, 5, 7, 10],
            'estimator__weights': ['uniform', 'distance'],
            'estimator__p': [1, 2]   # 1 = Manhattan, 2 = Euclidiana
        }
    
    X_train, y_train = crear_dataset_supervisado_knn(
        train_fold.values, train_fold.index, n_input=n_input, n_output=len(val_fold)
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    knn = KNeighborsRegressor()
    multi_knn = MultiOutputRegressor(knn)
    
    tscv = TimeSeriesSplit(n_splits=3)
    grid = GridSearchCV(multi_knn, param_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(X_train_scaled, y_train)
    
    mejor_modelo = grid.best_estimator_
    
    x_lags_pred = train_fold.values[-n_input:]
    fecha_fin_input_pred = train_fold.index[-1]
    tendencia_pred = 1.0
    mes_pred = fecha_fin_input_pred.month
    mes_sin_pred = np.sin(2 * np.pi * mes_pred / 12)
    mes_cos_pred = np.cos(2 * np.pi * mes_pred / 12)
    covid_pred = 1 if (val_fold.index[0] >= pd.Timestamp('2020-03-01') and 
                       val_fold.index[0] <= pd.Timestamp('2020-12-01')) else 0
    x_full_pred = np.concatenate([x_lags_pred, [tendencia_pred, mes_sin_pred, mes_cos_pred, covid_pred]])
    X_pred = x_full_pred.reshape(1, -1)
    X_pred_scaled = scaler.transform(X_pred)
    
    predicciones = mejor_modelo.predict(X_pred_scaled).flatten()
    
    real = val_fold.values
    pred = predicciones
    
    mse = mean_squared_error(real, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(real, pred)
    mape = np.mean(np.abs((real - pred) / np.where(real == 0, 1, real))) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred) + 1e-10)) * 100
    
    metricas = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape
    }
    
    return predicciones, metricas, mejor_modelo, scaler

In [48]:
resultados_cv_knn = []

print("-" * 50)
print("Validación cruzada - KNN Regression (C32)")
print("-" * 50)

for i, (train_fold, val_fold) in enumerate(ventanas_c32, 1):
    print(f"\n--- Fold {i} ---")
    
    predicciones, metricas, modelo, _ = evaluar_knn(train_fold, val_fold)
    
    best_params = modelo.get_params()
    n_neighbors = best_params['estimator__n_neighbors']
    weights = best_params['estimator__weights']
    p = best_params['estimator__p']
    
    resultados_cv_knn.append({
        'fold': i,
        'train_periodo': f"{train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}",
        'val_periodo': f"{val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}",
        'n_neighbors': n_neighbors,
        'weights': weights,
        'p': p,
        'metricas': metricas
    })
    
    print(f"Train: {train_fold.index.min().strftime('%Y-%m')} - {train_fold.index.max().strftime('%Y-%m')}")
    print(f"Val: {val_fold.index.min().strftime('%Y-%m')} - {val_fold.index.max().strftime('%Y-%m')}")
    print(f"Mejores parámetros: n_neighbors={n_neighbors}, weights={weights}, p={p}")
    print(f"RMSE: {metricas['RMSE']:.2f}")
    print(f"MAE: {metricas['MAE']:.2f}")
    print(f"MAPE: {metricas['MAPE']:.2f}%")
    print(f"SMAPE: {metricas['SMAPE']:.2f}%")
    print(f"MSE: {metricas['MSE']:.2f}")

metricas_promedio_knn = {
    'RMSE': np.mean([r['metricas']['RMSE'] for r in resultados_cv_knn]),
    'MAE': np.mean([r['metricas']['MAE'] for r in resultados_cv_knn]),
    'MAPE': np.mean([r['metricas']['MAPE'] for r in resultados_cv_knn]),
    'SMAPE': np.mean([r['metricas']['SMAPE'] for r in resultados_cv_knn]),
    'MSE': np.mean([r['metricas']['MSE'] for r in resultados_cv_knn])
}

print("\n" + "-" * 50)
print("Métricas promedio en validación cruzada")
print("-" * 50)
print(f"RMSE: {metricas_promedio_knn['RMSE']:.2f}")
print(f"MAE: {metricas_promedio_knn['MAE']:.2f}")
print(f"MAPE: {metricas_promedio_knn['MAPE']:.2f}%")
print(f"SMAPE: {metricas_promedio_knn['SMAPE']:.2f}%")
print(f"MSE: {metricas_promedio_knn['MSE']:.2f}")

--------------------------------------------------
Validación cruzada - KNN Regression (C32)
--------------------------------------------------

--- Fold 1 ---
Train: 2018-01 - 2020-12
Val: 2021-01 - 2021-12
Mejores parámetros: n_neighbors=2, weights=distance, p=1
RMSE: 30.16
MAE: 27.05
MAPE: 126.60%
SMAPE: 71.61%
MSE: 909.36

--- Fold 2 ---
Train: 2018-01 - 2021-12
Val: 2022-01 - 2022-12
Mejores parámetros: n_neighbors=7, weights=distance, p=1
RMSE: 20.36
MAE: 18.00
MAPE: 65.60%
SMAPE: 54.46%
MSE: 414.52

--- Fold 3 ---
Train: 2018-01 - 2022-12
Val: 2023-01 - 2023-12
Mejores parámetros: n_neighbors=7, weights=distance, p=2
RMSE: 13.86
MAE: 12.12
MAPE: 37.74%
SMAPE: 40.69%
MSE: 192.10

--- Fold 4 ---
Train: 2018-01 - 2023-12
Val: 2024-01 - 2024-12
Mejores parámetros: n_neighbors=10, weights=distance, p=2
RMSE: 19.50
MAE: 15.95
MAPE: 37.28%
SMAPE: 39.70%
MSE: 380.11

--------------------------------------------------
Métricas promedio en validación cruzada
------------------------------

In [49]:
print("-" * 50)
print("Evaluación final en Test (2025) - KNN Regression")
print("-" * 50)

X_train_full, y_train_full = crear_dataset_supervisado_knn(
    train_c32.values, train_c32.index, n_input=12, n_output=12
)

scaler_final = StandardScaler()
X_train_full_scaled = scaler_final.fit_transform(X_train_full)

param_grid_final = {
    'estimator__n_neighbors': [2, 3, 5, 7, 10, 12],
    'estimator__weights': ['uniform', 'distance'],
    'estimator__p': [1, 2]
}

knn_final = KNeighborsRegressor()
multi_knn_final = MultiOutputRegressor(knn_final)
tscv = TimeSeriesSplit(n_splits=3)
grid_final = GridSearchCV(multi_knn_final, param_grid_final, cv=tscv, 
                          scoring='neg_mean_squared_error', n_jobs=-1)
grid_final.fit(X_train_full_scaled, y_train_full)

modelo_final_knn = grid_final.best_estimator_
best_params_final = grid_final.best_params_
print(f"Mejores parámetros: {best_params_final}")

x_lags_pred_test = train_c32.values[-12:]
fecha_fin_pred_test = train_c32.index[-1]
tendencia_pred_test = 1.0
mes_pred_test = fecha_fin_pred_test.month
mes_sin_pred_test = np.sin(2 * np.pi * mes_pred_test / 12)
mes_cos_pred_test = np.cos(2 * np.pi * mes_pred_test / 12)
covid_pred_test = 0
x_full_pred_test = np.concatenate([x_lags_pred_test, [tendencia_pred_test, mes_sin_pred_test, mes_cos_pred_test, covid_pred_test]])
X_pred_test = x_full_pred_test.reshape(1, -1)
X_pred_test_scaled = scaler_final.transform(X_pred_test)

pred_test_knn = modelo_final_knn.predict(X_pred_test_scaled).flatten()

real_test = test_c32.values
pred_test = pred_test_knn

mse_test = mean_squared_error(real_test, pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(real_test, pred_test)
mape_test = np.mean(np.abs((real_test - pred_test) / np.where(real_test == 0, 1, real_test))) * 100
smape_test = np.mean(2 * np.abs(real_test - pred_test) / (np.abs(real_test) + np.abs(pred_test) + 1e-10)) * 100

print(f"RMSE: {rmse_test:.2f}")
print(f"MAE: {mae_test:.2f}")
print(f"MAPE: {mape_test:.2f}%")
print(f"SMAPE: {smape_test:.2f}%")
print(f"MSE: {mse_test:.2f}")

errores_test = test_c32.values - pred_test

--------------------------------------------------
Evaluación final en Test (2025) - KNN Regression
--------------------------------------------------
Mejores parámetros: {'estimator__n_neighbors': 12, 'estimator__p': 2, 'estimator__weights': 'distance'}
RMSE: 22.58
MAE: 20.73
MAPE: 251.80%
SMAPE: 79.80%
MSE: 509.95


In [50]:
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=train_c32.index, y=train_c32.values,
    mode='lines+markers', name='Train (2018-2024)',
    line=dict(color='cornflowerblue', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c32.index, y=test_c32.values,
    mode='lines+markers', name='Test Real (2025)',
    line=dict(color='green', width=2), marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=test_c32.index, y=pred_test_knn,
    mode='lines+markers', name='Predicción KNN',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(x='2025-01-01', line_width=2, line_dash='dash', line_color='darkgray', opacity=1)
fig1.add_annotation(x='2025-01-01', y=1, yref='paper', text='<b>Inicio Test</b>',
                    showarrow=False, font=dict(size=12, color='darkgray'),
                    xanchor='left', yanchor='bottom')

fig1.update_layout(
    title=dict(text=f'C32 - KNN Regression: Predicción vs Real<br>'
                    f'<sup>Test: RMSE={rmse_test:.1f} | MAE={mae_test:.1f} | MAPE={mape_test:.1f}%</sup>',
               x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig1.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses, y=test_c32.values,
    mode='lines+markers', name='Real',
    line=dict(color='green', width=2), marker=dict(size=4, symbol='circle'),
    hovertemplate='Mes: %{x}<br>Real: %{y:,.0f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=meses, y=pred_test_knn,
    mode='lines+markers', name='Predicción KNN',
    line=dict(color='red', width=2), marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='Zoom: Test 2025 (Predicho vs Real)', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.7)',
                bordercolor='lightgray', borderwidth=1),
    width=1055, height=500
)
fig2.show()

errores_test = test_c32.values - pred_test_knn

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=meses, y=errores_test,
    name='Error (Real - Predicción)',
    marker_color='mediumpurple', marker_line_color='darkorchid',
    marker_line_width=1, opacity=1,
    hovertemplate='Mes: %{x}<br>Error: %{y:,.0f} comparendos<extra></extra>'
))

fig3.add_hline(y=0, line_width=1.5, line_color='black', line_dash='solid', opacity=1)

fig3.update_layout(
    title=dict(text='Errores por mes (Real - Predicción) - Test 2025', x=0.5, font=dict(size=16, weight='bold')),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Error (Comparendos)', font=dict(weight='bold')),
    template='plotly_white', hovermode='x unified',
    showlegend=False, width=1055, height=500
)
fig3.show()

## Tabla 1 – Métricas Promedio en Validación Cruzada (CV)

| Modelo                     | RMSE   | MAE    | MAPE    | SMAPE   | MSE     |
|----------------------------|--------|--------|---------|---------|---------|
| Holt‑Winters (SES)         | 19.23  | 16.33  | 59.94%  | 47.60%  | 378.84  |
| ARIMA(1,0,0)               | 16.12  | 13.69  | 57.22%  | 40.75%  | 326.87  |
| SARIMA(1,0,1)(0,0,0)[12]   | 18.08  | 14.71  | 40.48%  | 49.98%  | 420.95  |
| Theta (θ=0.25)             | 19.38  | 16.07  | 57.62%  | 52.66%  | 2642.16 |
| Ridge                      | 17.63  | 15.16  | 57.59%  | 44.59%  | 322.23  |
| Lasso                      | 15.92  | 12.70  | 46.63%  | 40.49%  | 257.87  |
| Random Forest              | 19.51  | 17.25  | 64.44%  | 50.97%  | 390.92  |
| XGBoost                    | 20.54  | 17.53  | 66.52%  | 49.15%  | 454.11  |
| LightGBM                   | 18.39  | 16.05  | 62.73%  | 46.22%  | 356.90  |
| KNN                        | 20.97  | 18.28  | 66.81%  | 51.62%  | 474.02  |

## Tabla 2 – Métricas en Test (2025)

| Modelo                     | RMSE   | MAE    | MAPE    | SMAPE   | MSE     |
|----------------------------|--------|--------|---------|---------|---------|
| Holt‑Winters (SES)         | 18.07  | 16.15  | 207.37% | 70.24%  | 326.40  |
| ARIMA(1,0,0)               | 20.73  | 18.80  | 235.34% | 75.98%  | 429.56  |
| SARIMA(1,0,1)(0,0,0)[12]   | 7.87   | 6.61   | 88.69%  | 43.05%  | 61.98   |
| Theta (θ=0.25)             | 31.34  | 29.02  | 265.39% | 94.72%  | 981.92  |
| Ridge                      | 18.82  | 17.15  | 214.60% | 72.79%  | 354.37  |
| Lasso                      | 18.85  | 17.28  | 213.82% | 73.22%  | 355.25  |
| Random Forest              | 26.05  | 24.08  | 286.61% | 85.71%  | 678.79  |
| XGBoost                    | 21.97  | 20.40  | 240.94% | 79.64%  | 482.73  |
| LightGBM                   | 18.59  | 16.30  | 218.21% | 69.89%  | 345.44  |
| KNN                        | 22.58  | 20.73  | 251.80% | 79.80%  | 509.95  |

## Tabla 3 – Errores por Mes en Test (2025)

| Modelo                     | Ene | Feb | Mar | Abr | May | Jun | Jul | Ago | Sep | Oct | Nov | Dic |
|----------------------------|-----|-----|-----|-----|-----|-----|-----|-----|-----|-----|-----|-----|
| Holt‑Winters (SES)         | -5  | -16 | -13 | -13 | -13 | -13 | -6  | -12 | -18 | -28 | -31 | -28 |
| ARIMA(1,0,0)               | -5  | -17 | -15 | -15 | -16 | -16 | -9  | -15 | -21 | -31 | -34 | -31 |
| SARIMA(1,0,1)(0,0,0)[12]   | -2  | -11 | -7  | -5  | -4  | -2  | 6   | 1   | -4  | -12 | -14 | -10 |
| Theta (θ=0.25)             | -26 | -26 | -23 | -44 | -38 | -50 | -19 | 4   | -37 | -25 | -33 | -22 |
| Ridge                      | -8  | -18 | -14 | -14 | -13 | -14 | -7  | -12 | -18 | -28 | -32 | -29 |
| Lasso                      | -9  | -18 | -14 | -14 | -13 | -14 | -7  | -12 | -18 | -28 | -31 | -28 |
| Random Forest              | -6  | -17 | -14 | -18 | -22 | -29 | -22 | -18 | -29 | -36 | -42 | -35 |
| XGBoost                    | -6  | -20 | -16 | -14 | -21 | -24 | -16 | -15 | -16 | -34 | -33 | -30 |
| LightGBM                   | -4  | -14 | -12 | -11 | -12 | -14 | -7  | -12 | -18 | -28 | -35 | -29 |
| KNN                        | -3  | -14 | -13 | -19 | -19 | -23 | -16 | -18 | -24 | -31 | -38 | -31 |

## Interpretación de los Resultados de los Modelos

**a) Desempeño en validación cruzada (CV, 2018‑2024)**  
- Los modelos mostraron un RMSE promedio entre 16 y 21 comparendos, con diferencias relativamente pequeñas. Destacaron ARIMA(1,0,0) (RMSE 16.12) y Lasso (15.92).  
- El SARIMA(1,0,1)(0,0,0)[12] equivalente a un ARMA(1,1) obtuvo un RMSE en CV de 18.08, ligeramente superior.  
- Las métricas porcentuales (MAPE, SMAPE) fueron altas, influidas por el bajo volumen de la infracción (promedios mensuales de 25‑47 comparendos), donde pequeños errores absolutos generan porcentajes elevados.

**b) Desempeño en el conjunto de prueba (2025)**  
- El SARIMA(1,0,1)(0,0,0)[12] logró un rendimiento excepcional, con un RMSE de 7.87 y un MAE de 6.61, más del doble de precisión que cualquier otro modelo.  
- Todos los demás modelos fracasaron en 2025, con RMSE de 18 a 31, MAPE superiores al 200 % y errores sistemáticamente negativos (sobrestimación) en todos los meses.  
- Los errores mensuales del SARIMA fueron pequeños y alternaron signos (ej. enero -2, julio +6, agosto +1), indicando que sus predicciones siguieron de cerca los valores reales. El resto de modelos, en cambio, sobrestimaron constantemente, especialmente hacia finales de año (noviembre con errores de -31 a -42).

**c) Patrón general**  
- La realidad de 2025 fue una caída brusca en el volumen de infracciones C32, que ningún modelo excepto el SARIMA(1,0,1)(0,0,0)[12] pudo anticipar o adaptarse a ella.

---

## ¿Por qué se llegó a estos resultados?

**a) Características de la serie C32 (Paso de peatones)**  
- La serie es estacionaria y sin tendencia significativa (pendiente -0.04 comparendos/mes, R²=0.01, p=0.27), con una estacionalidad moderada (10.7 %).  
- Los residuos de la descomposición STL no son ruido blanco (Ljung‑Box p<0.001), lo que revela una estructura de dependencia temporal no capturada completamente por el modelo STL.  
- La presencia de valores atípicos y la alta variabilidad (coeficiente de variación cercano al 60‑80 % en muchos meses) hacen que la serie sea difícil de predecir.

**b) El quiebre de nivel en 2025**  
- En 2025, el número de infracciones se desplomó a niveles persistentemente más bajos que en todo el histórico. La media mensual de 2025 fue notablemente inferior a la media histórica, y la mayoría de los modelos basados en el nivel medio pasado sobrestimaron fuertemente.  
- Los modelos Holt‑Winters SES, ARIMA(1,0,0), Ridge y los basados en árboles proyectaron un nivel acorde con la media histórica o con una tendencia suave, sin capacidad de ajustarse rápidamente al nuevo régimen.

**c) ¿Por qué el SARIMA(1,0,1)(0,0,0)[12] (ARMA(1,1)) funcionó tan bien?**  
- El ARMA(1,1) combina un término autorregresivo (AR) con un promedio móvil (MA). La componente MA actúa como un “amortiguador” que permite que el modelo corrija sus pronósticos a partir de los errores recientes.  
- Al iniciar 2025 con valores más bajos de lo esperado, el MA(1) incorporó esos errores negativos en la predicción de los meses siguientes, bajando rápidamente el nivel pronosticado y ajustándose a la nueva realidad.  
- Los demás modelos, incluso el AR(1), no tienen esa flexibilidad de memoria corta; un AR(1) solo depende de los valores pasados, no de los errores, por lo que su adaptación es más lenta. Los métodos de suavizamiento exponencial simple también requieren varios períodos para ajustar el nivel, y la caída abrupta los desbordó.

**d) Lección**  
- Este es un caso clásico donde un modelo aparentemente similar en el pasado (AR(1) y ARMA(1,1) en CV eran parecidos) puede comportarse de manera radicalmente distinta ante un cambio estructural. La capacidad del MA para absorber shocks hizo la diferencia.

---

## Modelo Ideal para Predecir 2026

Se recomienda el modelo SARIMA(1,0,1)(0,0,0)[12] es decir, un ARMA(1,1) para el pronóstico de 2026 del código C32.

**Justificación:**  
- Fue el único modelo con desempeño aceptable en 2025 (RMSE 7.9 vs. el siguiente mejor 18.1). La magnitud de la ventaja excluye cualquier otra opción.  
- La componente de promedio móvil le permite adaptarse velozmente a cambios en el nivel medio, como el observado en 2025. Si el nuevo nivel más bajo se mantiene, el ARMA(1,1) lo proyectará correctamente; si la serie vuelve a subir, también se ajustará con rapidez.  
- Es un modelo eficiente (2 parámetros más la constante), fácil de reestimar cada mes al incorporar nuevos datos, lo que garantiza su vigencia.

## Pronóstico de la Infracción C32 para el Año 2026

Una vez seleccionado el mejor modelo predictivo (SARIMA con orden autorregresivo de primer orden y promedio móvil de primer orden (1,0,1), sin componente estacional estacional (0,0,0,12)), se procede a realizar el pronóstico del volumen de infracciones por no respetar el paso de peatones para los 12 meses del año 2026. El modelo se entrena con la serie completa de 96 meses (enero 2018 a diciembre 2025) y genera predicciones puntuales con sus respectivos intervalos de confianza para cada mes de 2026. Se imprime una tabla con los valores pronosticados mes a mes y se visualizan los resultados mediante dos gráficos interactivos: el primero muestra la serie histórica completa junto con la proyección hacia 2026, con una línea vertical discontinua que marca el inicio del período pronosticado; el segundo presenta exclusivamente el pronóstico mensual para 2026, facilitando la interpretación de la tendencia esperada mes a mes. Esta visualización permite anticipar el comportamiento futuro de la infracción y apoyar la toma de decisiones en materia de control de pasos peatonales y seguridad vial.

In [51]:
order_c32 = (1, 0, 1)
seasonal_order_c32 = (0, 0, 0, 12)

modelo_c32_2026 = SARIMAX(
    serie_c32,
    order=order_c32,
    seasonal_order=seasonal_order_c32,
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

forecast_2026_c32 = modelo_c32_2026.forecast(steps=12)

idx_forecast = pd.date_range(start='2026-01-01', periods=12, freq='MS')
forecast_2026_c32.index = idx_forecast

fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=serie_c32.index,
    y=serie_c32.values,
    mode='lines+markers',
    name='Histórico (2018-2025)',
    line=dict(color='cornflowerblue', width=2),
    marker=dict(size=4),
    hovertemplate='Fecha: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
))

fig1.add_trace(go.Scatter(
    x=forecast_2026_c32.index,
    y=forecast_2026_c32.values,
    mode='lines+markers',
    name='Predicción SARIMA(1,0,1)(0,0,0)[12]',
    line=dict(color='red', width=2),
    marker=dict(size=4, symbol='square'),
    hovertemplate='Fecha: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig1.add_vline(
    x='2026-01-01',
    line_width=2,
    line_dash='dash',
    line_color='darkgray',
    opacity=1
)

fig1.add_annotation(
    x='2026-01-01',
    y=1,
    yref='paper',
    text='<b>Inicio 2026</b>',
    showarrow=False,
    font=dict(size=12, color='darkgray'),
    xanchor='left',
    yanchor='bottom'
)

fig1.update_layout(
    title=dict(
        text='C32 - SARIMA(1,0,1)(0,0,0)[12]: Predicción 2026<br>'
             '<sup>Modelo entrenado con datos hasta diciembre 2025</sup>',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Fecha', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        x=0.01,
        y=0.99,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='lightgray',
        borderwidth=1
    ),
    width=1055,
    height=500
)

fig1.show()

meses_2026 = ['Enero','Febrero','Marzo','Abril','Mayo','Junio',
              'Julio','Agosto','Septiembre','Octubre','Noviembre','Diciembre']

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=meses_2026,
    y=forecast_2026_c32.values,
    mode='lines+markers',
    name='Predicción SARIMA(1,0,1)(0,0,0)[12]',
    line=dict(color='red', width=2),
    marker=dict(size=4, symbol='square'),
    hovertemplate='Mes: %{x}<br>Predicción: %{y:,.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(
        text='Pronóstico C32 para el año 2026',
        x=0.5,
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(text='Mes', font=dict(weight='bold')),
    yaxis_title=dict(text='Comparendos', font=dict(weight='bold')),
    template='plotly_white',
    hovermode='x unified',
    showlegend=False,
    width=1055,
    height=500
)

fig2.show()